# 🎙️ Open-Dubbing — AI Video Dubbing
Dubla vídeos automaticamente usando modelos de IA (Whisper, NLLB, TTS).

> Execute as células em ordem. Use GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
#@title 📦 1. Instalar dependências
#@markdown Execute esta célula uma vez por sessão.

!pip install -q open-dubbing
!apt-get install -q ffmpeg

import subprocess
result = subprocess.run(["ffprobe", "-version"], capture_output=True)
print("✅ ffmpeg instalado" if result.returncode == 0 else "❌ ffmpeg não encontrado")

import importlib
try:
    importlib.import_module("open_dubbing")
    print("✅ open-dubbing instalado")
except ImportError:
    print("❌ open-dubbing não encontrado")

In [ ]:
#@title ☁️ 2. Montar Google Drive (opcional)
#@markdown Monte para usar vídeos salvos no Drive ou salvar o resultado lá.

use_google_drive = True #@param {type:"boolean"}

if use_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive montado em /content/drive")
else:
    print("Drive não montado. Use o upload de arquivos do Colab.")

In [ ]:
#@title 🔑 3. Tokens e chaves de API
#@markdown O token do Hugging Face é **obrigatório** para o PyAnnote (diarização).
#@markdown Aceite os termos em: https://huggingface.co/pyannote/speaker-diarization-3.1

hugging_face_token = "" #@param {type:"string"}
openai_api_key = "" #@param {type:"string"}
#@markdown (OpenAI API Key só é necessária se usar TTS = openai)

import os
if hugging_face_token:
    os.environ["HF_TOKEN"] = hugging_face_token
    print("✅ HF_TOKEN definido")
else:
    print("⚠️ Hugging Face token não fornecido")

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key
    print("✅ OPENAI_API_KEY definido")

In [ ]:
#@title ⚙️ 4. Configurações do vídeo e idiomas

#@markdown ### Arquivo de entrada
input_file = "/content/video.mp4" #@param {type:"string"}
#@markdown Caminho para o vídeo. Se usou o Drive: `/content/drive/MyDrive/video.mp4`

output_directory = "/content/output" #@param {type:"string"}

#@markdown ---
#@markdown ### Idiomas
source_language = "eng" #@param {type:"string"}
#@markdown Idioma do vídeo original em ISO 639-3. Deixe vazio para autodetecção.

target_language = "por" #@param {type:"string"}
#@markdown Idioma de destino em ISO 639-3. Ex: `por` (Português), `spa` (Espanhol), `fra` (Francês)

target_language_region = "" #@param {type:"string"}
#@markdown Região para o TTS. Ex: `BR` para Português do Brasil. Deixe vazio para padrão.

In [ ]:
#@title 🧠 5. Configurações dos modelos de IA

#@markdown ### Speech-to-Text (Whisper)
stt = "faster-whisper" #@param ["auto", "faster-whisper", "transformers"]
#@markdown `faster-whisper` é mais rápido e recomendado no Colab com GPU.

whisper_model = "turbo" #@param ["medium", "large-v2", "large-v3", "turbo"]
#@markdown `turbo` = large-v3-turbo (destilado, ~4x mais rápido que large-v3, boa qualidade)

enable_vad = True #@param {type:"boolean"}
#@markdown VAD (Voice Activity Detection): reduz alucinações do Whisper. Recomendado.

#@markdown ---
#@markdown ### Tradução
translator = "nllb" #@param ["nllb", "apertium"]

nllb_model = "nllb-200-1.3B" #@param ["nllb-200-1.3B", "nllb-200-3.3B"]
#@markdown `nllb-200-1.3B` = mais rápido | `nllb-200-3.3B` = melhor qualidade

apertium_server = "" #@param {type:"string"}
#@markdown URL do servidor Apertium (só necessário se translator = apertium)

#@markdown ---
#@markdown ### Text-to-Speech
tts = "mms" #@param ["mms", "edge", "openai", "coqui", "cli", "api"]
#@markdown `mms` = Meta MMS, suporta +1100 idiomas. `edge` = Microsoft Edge TTS (boa qualidade).

#@markdown ---
#@markdown ### Hardware
device = "cuda" #@param ["cuda", "cpu"]
#@markdown Use `cuda` se a sessão tiver GPU (recomendado).

cpu_threads = 0 #@param {type:"integer"}
#@markdown Número de threads para CPU (0 = padrão automático)

In [ ]:
#@title 🎬 6. Opções de saída

original_subtitles = False #@param {type:"boolean"}
#@markdown Incorporar legendas originais no vídeo de saída

dubbed_subtitles = False #@param {type:"boolean"}
#@markdown Incorporar legendas dubladas no vídeo de saída

clean_intermediate_files = True #@param {type:"boolean"}
#@markdown Remover arquivos intermediários após o processo

log_level = "INFO" #@param ["DEBUG", "INFO", "WARNING", "ERROR"]

In [ ]:
#@title ✅ 7. Verificar configurações

import os

print("=" * 50)
print("CONFIGURAÇÕES")
print("=" * 50)
print(f"  Entrada     : {input_file}")
print(f"  Saída       : {output_directory}")
print(f"  Idioma orig : {source_language if source_language else '(autodetecção)'}")
print(f"  Idioma dest : {target_language}")
print(f"  Região TTS  : {target_language_region if target_language_region else '(padrão)'}")
print("---")
print(f"  STT         : {stt}")
print(f"  Modelo Whisper: {whisper_model}")
print(f"  VAD         : {enable_vad}")
print(f"  Tradutor    : {translator}")
if translator == "nllb":
    print(f"  NLLB model  : {nllb_model}")
print(f"  TTS         : {tts}")
print(f"  Device      : {device}")
print("---")
print(f"  HF Token    : {'✅ definido' if os.environ.get('HF_TOKEN') else '❌ NÃO definido'}")
print("=" * 50)

if not os.path.exists(input_file):
    print(f"⚠️  AVISO: arquivo de entrada não encontrado: {input_file}")
else:
    print(f"✅ Arquivo de entrada encontrado")

if device == "cuda":
    import subprocess
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if r.returncode == 0:
        gpu_line = [l for l in r.stdout.split("\n") if "MiB" in l]
        print(f"✅ GPU disponível: {gpu_line[0].strip() if gpu_line else 'detectada'}")
    else:
        print("⚠️  GPU não detectada. Considere mudar device para 'cpu' ou ativar GPU no Colab.")

In [ ]:
#@title 🚀 8. Executar dublagem
#@markdown Esta célula pode demorar vários minutos dependendo do tamanho do vídeo.

import subprocess
import sys

cmd = [
    "open-dubbing",
    "--input_file", input_file,
    "--output_directory", output_directory,
    "--target_language", target_language,
    "--hugging_face_token", os.environ.get("HF_TOKEN", ""),
    "--stt", stt,
    "--whisper_model", whisper_model,
    "--translator", translator,
    "--tts", tts,
    "--device", device,
    "--log_level", log_level,
]

if source_language:
    cmd += ["--source_language", source_language]

if target_language_region:
    cmd += ["--target_language_region", target_language_region]

if enable_vad:
    cmd += ["--vad"]

if translator == "nllb":
    cmd += ["--nllb_model", nllb_model]
elif translator == "apertium" and apertium_server:
    cmd += ["--apertium_server", apertium_server]

if cpu_threads > 0:
    cmd += ["--cpu_threads", str(cpu_threads)]

if clean_intermediate_files:
    cmd += ["--clean-intermediate-files"]

if original_subtitles:
    cmd += ["--original_subtitles"]

if dubbed_subtitles:
    cmd += ["--dubbed_subtitles"]

if tts == "openai" and os.environ.get("OPENAI_API_KEY"):
    cmd += ["--openai_api_key", os.environ["OPENAI_API_KEY"]]

print("Comando:", " ".join(cmd))
print("\n" + "=" * 50)
print("Iniciando...")
print("=" * 50 + "\n")

result = subprocess.run(cmd, text=True)

if result.returncode == 0:
    print("\n✅ Dublagem concluída com sucesso!")
    print(f"Arquivos salvos em: {output_directory}")
else:
    print(f"\n❌ Erro na dublagem (código {result.returncode})")
    print("Verifique o log acima para mais detalhes.")

In [ ]:
#@title 📥 9. Download do vídeo dublado
#@markdown Execute para baixar o resultado ou copiá-lo para o Drive.

import os
from google.colab import files

download_result = True #@param {type:"boolean"}
copy_to_drive = False #@param {type:"boolean"}
drive_destination = "/content/drive/MyDrive/dubbed/" #@param {type:"string"}

output_files = []
if os.path.exists(output_directory):
    for f in os.listdir(output_directory):
        if f.endswith(".mp4") and "dubbed" in f:
            output_files.append(os.path.join(output_directory, f))

if not output_files:
    all_files = [f for f in os.listdir(output_directory)] if os.path.exists(output_directory) else []
    mp4_files = [f for f in all_files if f.endswith(".mp4")]
    output_files = [os.path.join(output_directory, f) for f in mp4_files]

if output_files:
    print(f"Arquivos encontrados: {output_files}")

    if copy_to_drive:
        import shutil
        os.makedirs(drive_destination, exist_ok=True)
        for f in output_files:
            dest = os.path.join(drive_destination, os.path.basename(f))
            shutil.copy2(f, dest)
            print(f"✅ Copiado para Drive: {dest}")

    if download_result:
        for f in output_files:
            print(f"Baixando: {f}")
            files.download(f)
else:
    print("⚠️  Nenhum arquivo de saída encontrado.")
    if os.path.exists(output_directory):
        print("Conteúdo do diretório de saída:")
        for f in os.listdir(output_directory):
            print(f"  {f}")